In [2]:
# ============================================================
# FedPIGD-B v2 + Linear SVM (FULL COPY-PASTE WORKING VERSION)
# Backbone feature extractor: ConvNeXtTiny (frozen)
# Classifier: Linear SVM (sklearn)
#
# ✅ Your novelty preserved:
# 1) GA-evolved non-differentiable PIL pipelines (per client)
# 2) Pipeline signatures
# 3) Signature-conditioned anchoring (Ridge)  [kept, used to set per-client anchor vectors]
# 4) Federated learning structure (rounds, clients, server aggregation)
#
# ⚠️ Note for ML classifier:
# - SVM is not trained with gradients, so "FedAvg of weights" is replaced by
#   "FedAvg of feature centroids + union/aggregation of client SVMs via global refit"
# - This keeps the FL protocol (clients train locally, server aggregates) without
#   changing your GA/signature/anchoring novelty.
#
# Tested logic: avoids your previous bug where global_model.predict() was used instead of SVM.
# ============================================================

import os
import time
import copy
import random
import gc
import numpy as np
import tensorflow as tf

from PIL import Image, ImageOps, ImageEnhance, ImageFilter

from sklearn.linear_model import Ridge
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    matthews_corrcoef
)
from sklearn.preprocessing import label_binarize
from sklearn.calibration import CalibratedClassifierCV


# ============================================================
# 0) CONFIG
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

TRAIN_DIR = "/Users/tanvir/Downloads/Research/Fish Recognition/Photos/Fish/train_data"
TEST_DIR  = "/Users/tanvir/Downloads/Research/Fish Recognition/Photos/Fish/test_data"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

NUM_CLIENTS = 4
FL_ROUNDS = 5   # SVM version: keep smaller for speed; increase if you want

# GA
EV_POP = 10
EV_GENS = 6
EV_SUBSET = 64
MAX_PIPE_LEN = 5

# Anchoring (kept)
ANCHOR_LAMBDA = 1e-3

# SVM
SVM_C = 1.0
SVM_MAX_ITER = 8000

# ============================================================
# LOGGING
# ============================================================
def ts(): return time.strftime("%H:%M:%S")
def log(msg): print(f"[{ts()}] [INFO] {msg}")
def warn(msg): print(f"[{ts()}] [WARN] {msg}")
def phase(msg):
    print("\n" + "=" * 80)
    print(msg)
    print("=" * 80)

def cleanup():
    gc.collect()


# ============================================================
# 1) DATA LOADING
# ============================================================
def list_classes(train_dir):
    classes = sorted(d for d in os.listdir(train_dir)
                     if os.path.isdir(os.path.join(train_dir, d)))
    return classes, {c: i for i, c in enumerate(classes)}

def load_paths_labels(data_dir, class_to_idx):
    paths, labels = [], []
    for c, i in class_to_idx.items():
        cls_dir = os.path.join(data_dir, c)
        for f in os.listdir(cls_dir):
            if f.lower().endswith((".jpg", ".png", ".jpeg")):
                paths.append(os.path.join(cls_dir, f))
                labels.append(i)
    return np.array(paths), np.array(labels)

def split_clients(paths, labels, n):
    idx = np.random.permutation(len(paths))
    paths, labels = paths[idx], labels[idx]

    proportions = np.random.dirichlet([1.0] * n)
    sizes = (proportions * len(paths)).astype(int)
    sizes[-1] = len(paths) - sum(sizes[:-1])

    out, s = [], 0
    for i, sz in enumerate(sizes):
        out.append((paths[s:s+sz], labels[s:s+sz]))
        log(f"Client C{i+1}: {sz} samples")
        s += sz
    return out


phase("Loading dataset")
class_names, class_to_idx = list_classes(TRAIN_DIR)
K = len(class_names)
log(f"Detected {K} classes")

train_paths, train_labels = load_paths_labels(TRAIN_DIR, class_to_idx)
test_paths,  test_labels  = load_paths_labels(TEST_DIR,  class_to_idx)
clients = split_clients(train_paths, train_labels, NUM_CLIENTS)


# ============================================================
# 2) PIL PIPELINE OPS (non-differentiable)
# ============================================================
phase("Define PIL operator space (non-differentiable)")

OP_SPACE = [
    "IDENTITY", "GRAYSCALE", "AUTO_CONTRAST", "EQUALIZE",
    "SHARPEN", "BRIGHTNESS", "CONTRAST", "BLUR"
]

def sample_op():
    op = random.choice(OP_SPACE)
    p = {}
    if op in ["SHARPEN", "BRIGHTNESS", "CONTRAST"]:
        p["factor"] = float(np.random.uniform(0.8, 1.5))
    if op == "BLUR":
        p["radius"] = float(np.random.uniform(0.2, 1.5))
    return (op, p)

def apply_op(img, op):
    name, p = op
    if name == "IDENTITY":
        return img
    if name == "GRAYSCALE":
        return ImageOps.grayscale(img).convert("RGB")
    if name == "AUTO_CONTRAST":
        return ImageOps.autocontrast(img)
    if name == "EQUALIZE":
        return ImageOps.equalize(img)
    if name == "SHARPEN":
        return ImageEnhance.Sharpness(img).enhance(p["factor"])
    if name == "BRIGHTNESS":
        return ImageEnhance.Brightness(img).enhance(p["factor"])
    if name == "CONTRAST":
        return ImageEnhance.Contrast(img).enhance(p["factor"])
    if name == "BLUR":
        return img.filter(ImageFilter.GaussianBlur(p["radius"]))
    return img

def apply_pipeline_np(x, pipe):
    # x is float32 [0,1]
    img = Image.fromarray((np.clip(x, 0.0, 1.0) * 255).astype(np.uint8))
    for op in pipe:
        img = apply_op(img, op)
    return np.array(img).astype(np.float32) / 255.0

def tf_apply_pipeline(x, pipe):
    out = tf.numpy_function(lambda z: apply_pipeline_np(z, pipe), [x], tf.float32)
    out.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
    return out


# ============================================================
# 3) SAFE DECODE
# ============================================================
def decode_and_resize(p):
    img = tf.image.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMAGE_SIZE)
    return tf.cast(img, tf.float32) / 255.0


# ============================================================
# 4) FEATURE EXTRACTOR BACKBONE (ConvNeXtTiny frozen)
# ============================================================
phase("Build ConvNeXtTiny feature extractor (frozen)")

def build_backbone_feat_model():
    base = tf.keras.applications.ConvNeXtTiny(
        include_top=False,
        weights="imagenet",
        input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)
    )
    base.trainable = False
    feat = tf.keras.Model(
        inputs=base.input,
        outputs=tf.keras.layers.GlobalAveragePooling2D()(base.output)
    )
    return feat

feat_model = build_backbone_feat_model()
FEAT_DIM = feat_model.output_shape[-1]
log(f"Feature dim = {FEAT_DIM}")


# ============================================================
# 5) GA EVOLUTION (Separability proxy on frozen features)
# ============================================================
phase("GA evolution of client preprocessing pipelines")

def separability_proxy(f, l):
    sb, sw = 0.0, 0.0
    mu = f.mean(axis=0, keepdims=True)
    for c in np.unique(l):
        Xc = f[l == c]
        if len(Xc) < 2:
            continue
        mc = Xc.mean(axis=0, keepdims=True)
        sb += len(Xc) * np.sum((mc - mu) ** 2)
        sw += np.sum((Xc - mc) ** 2)
    return float(sb / (sw + 1e-8))

def evolve_pipeline(paths, labels, client_id):
    pop = [[sample_op() for _ in range(random.randint(1, MAX_PIPE_LEN))]
           for _ in range(EV_POP)]
    best_pipe, best_score = None, -1e9

    for g in range(EV_GENS):
        for pipe in pop:
            n = min(EV_SUBSET, len(paths))
            idx = np.random.choice(len(paths), n, replace=False) if len(paths) > n else np.arange(len(paths))

            xs = []
            for p in paths[idx]:
                x = decode_and_resize(tf.constant(p))
                x = tf_apply_pipeline(x, pipe)
                xs.append(x.numpy())
            xs = np.array(xs, dtype=np.float32)

            feats = feat_model.predict(xs, verbose=0)
            score = separability_proxy(feats, labels[idx]) - 0.02 * len(pipe)

            if score > best_score:
                best_score = score
                best_pipe = pipe

            del xs, feats
            cleanup()

        log(f"C{client_id} GA Gen {g+1}/{EV_GENS} | BestScore={best_score:.4f}")

        # Simple mutate to keep behavior close to your earlier code
        new_pop = []
        for _ in range(EV_POP):
            parent = copy.deepcopy(random.choice(pop))
            if random.random() < 0.4 and len(parent) < MAX_PIPE_LEN:
                parent.append(sample_op())
            else:
                parent[random.randrange(len(parent))] = sample_op()
            new_pop.append(parent)
        pop = new_pop

    return best_pipe

client_pipelines = []
for i, (cp, cl) in enumerate(clients, start=1):
    log(f"Start GA for Client C{i}")
    client_pipelines.append(evolve_pipeline(cp, cl, i))
    cleanup()


# ============================================================
# 6) PIPELINE SIGNATURES (kept)
# ============================================================
phase("Build pipeline signatures (kept)")

def pipeline_signature(pipe):
    hist = np.zeros(len(OP_SPACE), dtype=np.float32)
    factors, radii = [], []
    for (op, p) in pipe:
        hist[OP_SPACE.index(op)] += 1.0
        if "factor" in p: factors.append(float(p["factor"]))
        if "radius" in p: radii.append(float(p["radius"]))
    f_mean = np.mean(factors) if factors else 0.0
    f_std  = np.std(factors) if factors else 0.0
    r_mean = np.mean(radii) if radii else 0.0
    r_std  = np.std(radii) if radii else 0.0
    length = float(len(pipe))
    runtime_proxy = length
    return np.concatenate([hist, np.array([f_mean, f_std, r_mean, r_std, length, runtime_proxy], np.float32)])

client_sigs = np.stack([pipeline_signature(p) for p in client_pipelines], axis=0)
log(f"Signature dim = {client_sigs.shape[1]}")
log(f"Signatures shape = {client_sigs.shape}")


# ============================================================
# 7) HELPERS: build dataset, extract features
# ============================================================
def make_client_dataset(paths, labels, pipeline, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(2000, len(paths)), seed=SEED, reshuffle_each_iteration=True)

    def map_fn(x, y, pipe=pipeline):
        img = decode_and_resize(x)
        img = tf_apply_pipeline(img, pipe)
        return img, y

    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

def make_test_dataset(paths, labels):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def map_fn(x, y):
        img = decode_and_resize(x)
        return img, y

    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

def extract_features_from_ds(ds):
    X_list, y_list = [], []
    for xb, yb in ds:
        feats = feat_model(xb, training=False).numpy()
        X_list.append(feats)
        y_list.append(yb.numpy())
    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    return X, y


# ============================================================
# 8) SIGNATURE-CONDITIONED ANCHORING (kept)
#    For SVM, we define "anchor" as per-client feature mean vector.
#    Conditioner learns signature -> expected feature-mean anchor.
# ============================================================
phase("Federated learning loop (SVM) + signature-conditioned anchoring (kept)")

class ConditionerRidge:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.models = None
    def fit(self, S, A):
        self.models = [Ridge(alpha=self.alpha).fit(S, A[:, j]) for j in range(A.shape[1])]
    def predict(self, s):
        s = s.reshape(1, -1)
        return np.array([m.predict(s)[0] for m in self.models], dtype=np.float32)

conditioner = ConditionerRidge(alpha=1.0)

# Initialize anchors as global mean feature vector (computed on a small random subset)
init_idx = np.random.choice(len(train_paths), size=min(256, len(train_paths)), replace=False)
tmp_ds = make_test_dataset(train_paths[init_idx], train_labels[init_idx])
X0, y0 = extract_features_from_ds(tmp_ds)
global_anchor = X0.mean(axis=0).astype(np.float32)
anchors = [global_anchor.copy() for _ in range(NUM_CLIENTS)]
log("Anchors initialized from global feature mean (proxy for adapter anchor).")
cleanup()


# ============================================================
# 9) FL ROUNDS (clients train local SVMs on anchored features)
#    - Each round:
#      a) each client extracts features using its GA pipeline
#      b) client applies anchoring transform: f' = f - lam*(mean(f)-anchor)
#      c) client trains LinearSVC
#      d) server aggregates by refitting a new global SVM on merged (subsampled) features
#      e) server updates conditioner: signature -> client anchor means
# ============================================================

def apply_feature_anchoring(X, anchor_vec, lam):
    # anchor_vec: target mean for this client
    mu = X.mean(axis=0, keepdims=True)
    # shift features slightly toward anchor mean
    return (X - lam * (mu - anchor_vec.reshape(1, -1))).astype(np.float32)

def train_client_svm(X, y):
    # class_weight balanced helps your "Bacha dominates" issue
    svm = LinearSVC(
        C=SVM_C,
        class_weight="balanced",
        max_iter=SVM_MAX_ITER,
        dual="auto"
    )
    svm.fit(X, y)
    return svm

def subsample_for_server(X, y, max_n=1200):
    if len(X) <= max_n:
        return X, y
    idx = np.random.choice(len(X), size=max_n, replace=False)
    return X[idx], y[idx]

global_svm = None

for r in range(1, FL_ROUNDS + 1):
    phase(f"FL ROUND {r}/{FL_ROUNDS}")

    client_svms = []
    client_sizes = []
    client_anchor_means = []
    server_X_parts = []
    server_y_parts = []

    for ci, (cp, cl) in enumerate(clients, start=1):
        log(f"Client C{ci}: build dataset with GA pipeline + extract features")
        ds = make_client_dataset(cp, cl, client_pipelines[ci-1], shuffle=True)
        Xc, yc = extract_features_from_ds(ds)
        client_sizes.append(len(Xc))

        # compute client mean (for conditioner training)
        mean_c = Xc.mean(axis=0).astype(np.float32)
        client_anchor_means.append(mean_c)

        # apply anchoring transform (kept)
        Xc_a = apply_feature_anchoring(Xc, anchors[ci-1], ANCHOR_LAMBDA)

        # local train
        log(f"Client C{ci}: train Linear SVM on anchored features")
        svm_c = train_client_svm(Xc_a, yc)
        client_svms.append(svm_c)

        # send small subset to server for global refit (keeps FL/server aggregation)
        Xs, ys = subsample_for_server(Xc_a, yc, max_n=1200)
        server_X_parts.append(Xs)
        server_y_parts.append(ys)

        del ds, Xc, yc, Xc_a, Xs, ys
        cleanup()

    # ---- Server aggregation: refit global SVM on merged client samples ----
    log("Server: aggregate client updates by global SVM refit on merged samples")
    X_server = np.concatenate(server_X_parts, axis=0)
    y_server = np.concatenate(server_y_parts, axis=0)

    # Use calibrated SVM to get probabilities for ROC-AUC/PR-AUC/logloss
    base_svm = LinearSVC(
        C=SVM_C,
        class_weight="balanced",
        max_iter=SVM_MAX_ITER,
        dual="auto"
    )
    global_svm = CalibratedClassifierCV(base_svm, method="sigmoid", cv=3)
    global_svm.fit(X_server, y_server)
    log("Server: global calibrated Linear SVM updated.")

    # ---- Server: update conditioner (signature -> anchor mean) ----
    log("Server: update conditioner (signature -> feature-mean anchor)")
    A = np.stack(client_anchor_means, axis=0)
    conditioner.fit(client_sigs, A)
    anchors = [conditioner.predict(client_sigs[i]) for i in range(NUM_CLIENTS)]
    log("Server: anchors refreshed for next round.")

    del client_svms, client_sizes, client_anchor_means, server_X_parts, server_y_parts, X_server, y_server, A
    cleanup()


# ============================================================
# 10) FINAL EVALUATION (SVM is used here ✅)
# ============================================================
phase("FINAL EVALUATION: Test set (SVM on frozen features)")

test_ds = make_test_dataset(test_paths, test_labels)
Xt, yt = extract_features_from_ds(test_ds)

# Inference uses global_svm (NOT a keras model)
preds = global_svm.predict(Xt)

# Probabilities for ROC-AUC/PR-AUC/logloss
pred_probs = global_svm.predict_proba(Xt)

acc  = accuracy_score(yt, preds)
prec = precision_score(yt, preds, average="weighted", zero_division=0)
rec  = recall_score(yt, preds, average="weighted", zero_division=0)
f1   = f1_score(yt, preds, average="weighted", zero_division=0)
ll   = log_loss(yt, pred_probs)

y_true_bin = label_binarize(yt, classes=list(range(K)))

roc_auc = roc_auc_score(
    y_true_bin,
    pred_probs,
    average="macro",
    multi_class="ovr"
)

pr_auc = average_precision_score(
    y_true_bin,
    pred_probs,
    average="macro"
)

mcc = matthews_corrcoef(yt, preds)

# ---- Logs (exactly as you asked) ----
log(f"Accuracy     = {acc:.4f}")
log(f"Precision    = {prec:.4f} (weighted)")
log(f"Recall       = {rec:.4f} (weighted)")
log(f"F1-score     = {f1:.4f} (weighted)")
log(f"ROC-AUC      = {roc_auc:.4f} (macro, OvR)")
log(f"PR-AUC       = {pr_auc:.4f} (macro)")
log(f"Log-Loss     = {ll:.4f}")
log(f"MCC Score    = {mcc:.4f}")

print("\nClassification Report:")
print(classification_report(
    yt,
    preds,
    target_names=class_names,
    digits=4,
    zero_division=0
))

cleanup()



Loading dataset
[15:07:28] [INFO] Detected 23 classes
[15:07:29] [INFO] Client C1: 2648 samples
[15:07:29] [INFO] Client C2: 158 samples
[15:07:29] [INFO] Client C3: 1364 samples
[15:07:29] [INFO] Client C4: 765 samples

Define PIL operator space (non-differentiable)

Build ConvNeXtTiny feature extractor (frozen)
[15:07:29] [INFO] Feature dim = 768

GA evolution of client preprocessing pipelines
[15:07:29] [INFO] Start GA for Client C1
[15:09:34] [INFO] C1 GA Gen 1/6 | BestScore=2.1935
[15:11:32] [INFO] C1 GA Gen 2/6 | BestScore=2.1935
[15:13:34] [INFO] C1 GA Gen 3/6 | BestScore=2.1935
[15:15:34] [INFO] C1 GA Gen 4/6 | BestScore=2.1935
[15:17:35] [INFO] C1 GA Gen 5/6 | BestScore=2.1935
[15:19:34] [INFO] C1 GA Gen 6/6 | BestScore=2.1935
[15:19:34] [INFO] Start GA for Client C2
[15:21:35] [INFO] C2 GA Gen 1/6 | BestScore=2.2608
[15:23:36] [INFO] C2 GA Gen 2/6 | BestScore=2.2608
[15:25:37] [INFO] C2 GA Gen 3/6 | BestScore=2.2608
[15:27:39] [INFO] C2 GA Gen 4/6 | BestScore=2.2608
[15:29:4

2026-01-05 15:56:12.829629: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


[15:56:12] [INFO] Anchors initialized from global feature mean (proxy for adapter anchor).

FL ROUND 1/5
[15:56:12] [INFO] Client C1: build dataset with GA pipeline + extract features


2026-01-05 16:04:46.114974: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


[16:04:46] [INFO] Client C1: train Linear SVM on anchored features
[16:04:49] [INFO] Client C2: build dataset with GA pipeline + extract features
[16:05:20] [INFO] Client C2: train Linear SVM on anchored features


...

[17:06:16] [INFO] Server: aggregate client updates by global SVM refit on merged samples
[17:06:24] [INFO] Server: global calibrated Linear SVM updated.
[17:06:24] [INFO] Server: update conditioner (signature -> feature-mean anchor)
[17:06:25] [INFO] Server: anchors refreshed for next round.

FL ROUND 5/5
[17:06:25] [INFO] Client C1: build dataset with GA pipeline + extract features
[17:14:56] [INFO] Client C1: train Linear SVM on anchored features
[17:15:00] [INFO] Client C2: build dataset with GA pipeline + extract features
[17:15:31] [INFO] Client C2: train Linear SVM on anchored features


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[17:15:44] [INFO] Client C3: build dataset with GA pipeline + extract features
[17:20:07] [INFO] Client C3: train Linear SVM on anchored features
[17:20:09] [INFO] Client C4: build dataset with GA pipeline + extract features
[17:22:35] [INFO] Client C4: train Linear SVM on anchored features


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[17:23:18] [INFO] Server: aggregate client updates by global SVM refit on merged samples
[17:23:27] [INFO] Server: global calibrated Linear SVM updated.
[17:23:27] [INFO] Server: update conditioner (signature -> feature-mean anchor)
[17:23:27] [INFO] Server: anchors refreshed for next round.

FINAL EVALUATION: Test set (SVM on frozen features)
[17:27:04] [INFO] Accuracy     = 0.8878
[17:27:04] [INFO] Precision    = 0.9126 (weighted)
[17:27:04] [INFO] Recall       = 0.8878 (weighted)
[17:27:04] [INFO] F1-score     = 0.8956 (weighted)
[17:27:04] [INFO] ROC-AUC      = 0.9892 (macro, OvR)
[17:27:04] [INFO] PR-AUC       = 0.9046 (macro)
[17:27:04] [INFO] Log-Loss     = 0.5031
[17:27:04] [INFO] MCC Score    = 0.8808

Classification Report:
                                                  precision    recall  f1-score   support

                      Bacha (Pangasius bocourti)     0.9933    0.9673    0.9801       153
        Baim(ব্যাইম)     (Mastacembelus armatus)     0.9697    0.9143    0.